# Semana 22 · ¿Cómo se pesa un planeta?

Este cuaderno acompaña a la sesión de proyecto de la semana 22: calcular masas con la tercera ley de Kepler y recuperar la masa de Júpiter a partir de observaciones de una de sus lunas.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. La balanza de Kepler

$$M = \frac{4\pi^2 r^3}{G\,T^2}$$

convirtiendo antes todo a metros y segundos. Los datos de las órbitas son valores redondeados; los de las lunas de Júpiter, los periodos contrastados y los radios de memoria.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

G = 6.674e-11
DIA = 86400.0

def masa_central(r_km, T_dias):
    r = r_km * 1e3
    T = T_dias * DIA
    return 4 * np.pi**2 * r**3 / (G * T**2)

sistemas = [   # (cuerpo central, satélite, radio de la órbita en km, periodo en días)
    ("Sol",     "Tierra",   1.496e8,   365.256),
    ("Tierra",  "Luna",     384_400,   27.322),
    ("Marte",   "Fobos",    9_376,     0.3189),
    ("Júpiter", "Ío",       421_700,   1.769),
    ("Júpiter", "Europa",   671_100,   3.551),
    ("Júpiter", "Ganimedes", 1_070_400, 7.155),
    ("Júpiter", "Calisto",  1_882_700, 16.689),
    ("Saturno", "Titán",    1_221_870, 15.945),
]
for central, sat, r, T in sistemas:
    print(f"{central:8s} pesado con {sat:9s}: M = {masa_central(r, T):.3e} kg")

Las cuatro lunas de Júpiter dan la misma masa con tres cifras. La de la Tierra sale un 1 % alta porque, como viste en la sesión 2, lo que se mide es la masa de la Tierra más la de la Luna.

## 2. La recta de las lunas de Júpiter

Si $T^2 = \dfrac{4\pi^2}{GM}\,r^3$, al dibujar $T^2$ frente a $r^3$ los puntos quedan en una recta que pasa por el origen, de pendiente $k = 4\pi^2/(GM)$. Midiendo la pendiente, $M = 4\pi^2/(G k)$. Con varias lunas, el ajuste promedia los errores de cada una.

In [ ]:
lunas = [s for s in sistemas if s[0] == "Júpiter"]
r3 = np.array([(s[2] * 1e3) ** 3 for s in lunas])
T2 = np.array([(s[3] * DIA) ** 2 for s in lunas])

# Recta por el origen: la pendiente que mejor ajusta es sum(x·y) / sum(x²)
k = np.sum(r3 * T2) / np.sum(r3**2)
M_jup = 4 * np.pi**2 / (G * k)
print(f"Pendiente: {k:.4e} s²/m³   →   masa de Júpiter: {M_jup:.4e} kg  ({M_jup / 5.972e24:.0f} masas terrestres)")

x = np.linspace(0, r3.max() * 1.05, 10)
plt.plot(r3, T2, "o")
for s, xi, yi in zip(lunas, r3, T2):
    plt.annotate(s[1], (xi, yi), textcoords="offset points", xytext=(-10, 8))
plt.plot(x, k * x, "--")
plt.xlabel("r³ (m³)"); plt.ylabel("T² (s²)"); plt.title("Lunas galileanas: T² frente a r³")
plt.grid(alpha=0.3); plt.show()

## 3. Unas observaciones con prismáticos

Con prismáticos no ves la órbita de Calisto: ves un punto que, noche tras noche, se aleja de Júpiter por un lado, vuelve, cruza y se aleja por el otro. Vista de canto, la órbita circular se convierte en un vaivén: la separación aparente sigue una función seno, $x(t) = A\sin(2\pi(t - t_0)/T)$, donde $A$ es la separación máxima.

Vamos a **fabricar** unas observaciones realistas: una medida cada noche durante 50 días, algunas noches nubladas (sin dato) y un error de medida de unos 0,7 minutos de arco, el tamaño del disco de Júpiter en los prismáticos. Fijamos la semilla del generador de números aleatorios para que a todos nos salgan los mismos datos.

In [ ]:
rng = np.random.default_rng(1610)          # semilla: el año del descubrimiento
T_real = 16.689                             # días
A_real = 10.3                               # minutos de arco (con Júpiter a 4,2 ua)
noches = np.arange(50.0)
despejadas = rng.random(50) > 0.3           # un 30 % de noches nubladas
t_obs = noches[despejadas]
x_obs = A_real * np.sin(2 * np.pi * (t_obs - 3.0) / T_real) + rng.normal(0, 0.7, t_obs.size)

plt.figure(figsize=(10, 3.5))
plt.plot(t_obs, x_obs, "o")
plt.axhline(0, color="gray", lw=0.5)
plt.xlabel("noche"); plt.ylabel("separación de Júpiter (′)\n(+ este, − oeste)")
plt.title("Calisto: observaciones simuladas"); plt.grid(alpha=0.3)
plt.show()

## 4. Recuperar el periodo y la amplitud

Ahora olvidamos los valores reales y los buscamos en los datos. La idea:

1. Para cada periodo de prueba $T$ entre 10 y 25 días, buscamos la mejor curva $a\sin(2\pi t/T) + b\cos(2\pi t/T)$, que es lo mismo que un seno con amplitud y desfase libres. Para $T$ fijo, es un problema lineal en $a$ y $b$, y `np.linalg.lstsq` lo resuelve por mínimos cuadrados.
2. Calculamos cuánto se separan los datos de esa curva (la suma de los cuadrados de los residuos).
3. Nos quedamos con el periodo que da la menor separación.

In [ ]:
def ajuste(T, t, x):
    """Mejor seno de periodo T. Devuelve (suma de residuos al cuadrado, amplitud)."""
    X = np.column_stack([np.sin(2 * np.pi * t / T), np.cos(2 * np.pi * t / T)])
    coef, res, *_ = np.linalg.lstsq(X, x, rcond=None)
    residuos = np.sum((x - X @ coef) ** 2)
    return residuos, np.hypot(*coef)

periodos = np.arange(10, 25, 0.005)
resultados = np.array([ajuste(T, t_obs, x_obs) for T in periodos])
i = np.argmin(resultados[:, 0])
T_med, A_med = periodos[i], resultados[i, 1]
print(f"Periodo medido: {T_med:.2f} días (real: {T_real})   amplitud medida: {A_med:.2f}′ (real: {A_real}′)")

plt.figure(figsize=(9, 3))
plt.plot(periodos, resultados[:, 0])
plt.axvline(T_med, color="r", lw=0.8)
plt.xlabel("periodo de prueba (días)"); plt.ylabel("suma de residuos²")
plt.title("El mejor periodo es el que deja menos residuos"); plt.grid(alpha=0.3)
plt.show()

**Ejercicio.** Convierte la amplitud medida en un radio de órbita (con $r = D\,\theta$, $D = 4{,}2$ ua y $\theta$ en radianes) y calcula la masa de Júpiter con el periodo medido. ¿Qué error tiene respecto a $1{,}898 \times 10^{27}$ kg?

In [ ]:
# Tu código aquí

In [ ]:
# Solución
D_km = 4.2 * 1.496e8
theta = np.radians(A_med / 60)
r_km = D_km * theta
M = masa_central(r_km, T_med)
print(f"radio de la órbita: {r_km:,.0f} km   masa de Júpiter: {M:.3e} kg   error: {100 * (M / 1.898e27 - 1):+.1f} %")

Fíjate en que el error de la masa es unas tres veces el error relativo del radio, porque el radio va al cubo: un 2 % de error en el ángulo da un 6 % en la masa. Medir bien la amplitud es lo que más importa.

## 5. Experimenta

1. Aumenta el error de medida a 2′. ¿Sigue saliendo bien el periodo? ¿Y la masa?
2. Reduce las observaciones a 15 noches. ¿Qué pasa con el periodo? ¿Por qué hacen falta observaciones que cubran varias vueltas?
3. Repite el proceso con Ío (periodo 1,769 días, amplitud unos 2,3′). Con una sola observación por noche, ¿qué periodo encuentras? Busca qué es el *aliasing*: si observas algo cada 24 horas, no puedes distinguir un periodo de 1,769 días de otros que encajan igual de bien con tus muestras.

In [ ]:
# Tu experimento aquí